# Data Cleaning and Model Training for SmartVoyageAI

This notebook focuses on cleaning the generated traveler data and training a model for user segmentation or recommendation.

In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import joblib
import os

## 1. Load the Data

We will load the preprocessed users data generated by the `datagenerator.py` script.

In [ ]:
# Adjust path if necessary
data_path = '../../data/Preprocessed_Users.csv'
df = pd.read_csv(data_path)

print(f"Data loaded successfully with {df.shape[0]} rows and {df.shape[1]} columns.")
df.head()

## 2. Data Cleaning

We will check for null values and duplicated entries.

In [ ]:
print("Checking for Null Values:")
print(df.isnull().sum())

print("\nChecking for Duplicate Values:")
print(f"Number of duplicated rows: {df.duplicated().sum()}")

### Handle Nulls or Duplicates if any

In [ ]:
# Drop duplicates if any
if df.duplicated().sum() > 0:
    df = df.drop_duplicates()
    print("Duplicates removed.")

# Handle nulls (e.g., fill with mean or drop)
if df.isnull().values.any():
    # Select only numeric columns for fillna(mean)
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].mean())
    print("Null values handled.")

## 3. Data Preprocessing for Training

We'll scale the scores and indices to prepare for clustering.

In [ ]:
features = ['Budget_Index', 'Adventure_Score', 'Culture_Score', 'Foodie_Score', 'Family_Focus']
X = df[features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

## 4. Model Training: KMeans Clustering

We use KMeans to segment users into different travel profiles.

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42)
df['Cluster'] = kmeans.fit_predict(X_scaled)

print("Model trained and clusters assigned.")

## 5. Visualizing Clusters

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='Adventure_Score', y='Culture_Score', hue='Cluster', palette='viridis')
plt.title('User Segments based on Adventure and Culture Scores')
plt.show()

## 6. Analyze Cluster Profiles

Understanding what each cluster represents by calculating the mean values of the features.

In [9]:
# Calculate the mean values for each cluster to understand the profiles
cluster_profiles = df.groupby('Cluster')[features].mean()
print("Cluster Profiles (Mean Scores):")
cluster_profiles

Cluster Profiles (Mean Scores):


,Budget_Index,Adventure_Score,Culture_Score,Foodie_Score,Family_Focus
Cluster,,,,,
0,0.429246,22.227612,33.186567,44.317164,4.406716
1,0.472712,70.162393,50.521368,52.333333,1.529915
2,0.837251,55.129825,48.182456,48.901754,5.298246
3,0.376948,55.253521,74.000000,51.352113,7.051643


## 7. Save the Model and Scaler

Saving the trained model for use in the Backend API.

In [14]:
# Create models directory if it doesn't exist
os.makedirs('../models', exist_ok=True)

# Save the model and scaler
joblib.dump(kmeans, '../models/user_segmentation_model.pkl')
joblib.dump(scaler, '../models/scaler.pkl')

print("Model and Scaler saved successfully in backend/ml/models/")

Model and Scaler saved successfully in backend/ml/models/
